# 02 — Build County Adjacency Matrix

Constructs a spatial weights matrix (queen contiguity) for US counties using
`libpysal`. The resulting adjacency matrix is used as the ICAR prior structure
in the BYM2 model (`04_bym2_model.py`).

**Outputs:**
- `analysis/data/adjacency_W.npz` — sparse scipy CSR adjacency matrix
- `analysis/data/node_order.csv`  — geo_id → node_idx mapping

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..'))

import numpy as np
import pandas as pd
import geopandas as gpd
import scipy.sparse as sp
import libpysal
from libpysal.weights import Queen

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
# Load county shapefile (update path if running locally)
import sys; sys.path.insert(0, '../scripts')
from config import SHAPEFILE_PATH

gdf = gpd.read_file(SHAPEFILE_PATH)[['GEOID', 'geometry']].rename(columns={'GEOID': 'geo_id'})
gdf = gdf.reset_index(drop=True)
print(f'Loaded {len(gdf):,} counties')

In [ ]:
# Build queen contiguity weights
print('Building queen contiguity weights (this may take a minute) ...')
W = Queen.from_dataframe(gdf, silence_warnings=True)
W.transform = 'B'  # binary (0/1) adjacency
print(f'Islands (disconnected): {W.islands}')
print(f'Mean neighbors: {W.mean_neighbors:.1f}')

# Convert to scipy sparse CSR
W_sparse = W.sparse.tocsr()
print(f'Adjacency matrix: {W_sparse.shape}, {W_sparse.nnz:,} non-zeros')

In [ ]:
# Save adjacency matrix and node order
sp.save_npz(os.path.join(DATA_DIR, 'adjacency_W.npz'), W_sparse)
print(f'Saved adjacency_W.npz')

node_order = gdf[['geo_id']].reset_index().rename(columns={'index': 'node_idx'})
node_order.to_csv(os.path.join(DATA_DIR, 'node_order.csv'), index=False)
print(f'Saved node_order.csv ({len(node_order):,} rows)')